In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    LongType,
    DoubleType,
    BooleanType
)

# Unity Catalog schemas
BRONZE_SCHEMA = "online_retail.indian_ecommerce_bronze"
SILVER_SCHEMA = "online_retail.indian_ecommerce_silver"

# Create Silver schema if it does not exist
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}
""")

# Source tables
TABLES = [
    "orders",
    "order_items",
    "customers",
    "products",
    "payments",
    "shipments",
    "returns",
    "customer_reviews",
    "marketing_campaigns"
]

print("Silver transformation environment initialized.")
print("Source:", BRONZE_SCHEMA)
print("Target:", SILVER_SCHEMA)


Silver transformation environment initialized.
Source: online_retail.indian_ecommerce_bronze
Target: online_retail.indian_ecommerce_silver


In [0]:

# Load Bronze tables into a dictionary

bronze_tables = {}

for table_name in TABLES:

    full_table_name = f"{BRONZE_SCHEMA}.{table_name}"

    bronze_tables[table_name] = spark.table(full_table_name)

    print(
        f"{table_name}: "
        f"{bronze_tables[table_name].count()} records"
    )

print("\nAll Bronze tables loaded successfully.")


orders: 100000 records
order_items: 254331 records
customers: 25000 records
products: 550 records
payments: 100000 records
shipments: 89681 records
returns: 12075 records
customer_reviews: 77530 records
marketing_campaigns: 151 records

All Bronze tables loaded successfully.


In [0]:

# Inspect source schemas

for table_name, df in bronze_tables.items():

    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name.upper()}")
    print(f"{'=' * 60}")

    df.printSchema()



TABLE: ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_time: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_method: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- shipping_fee: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- final_amount: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)


TABLE: ORDER_ITEMS
root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string 

In [0]:

# Calculate null counts for every source column

quality_results = []

for table_name, df in bronze_tables.items():

    total_rows = df.count()

    for column_name in df.columns:

        null_count = df.filter(
            F.col(column_name).isNull()
        ).count()

        quality_results.append((
            table_name,
            column_name,
            total_rows,
            null_count
        ))

quality_df = spark.createDataFrame(
    quality_results,
    [
        "table_name",
        "column_name",
        "total_rows",
        "null_count"
    ]
)

display(
    quality_df
    .filter(F.col("null_count") > 0)
    .orderBy("table_name", "column_name")
)


table_name,column_name,total_rows,null_count
customer_reviews,review_sentiment,77530,233
customers,age,25000,250
customers,last_order_date,25000,2300
customers,preferred_device,25000,125
orders,campaign_id,100000,82821
orders,coupon_code,100000,33711
products,rating_average,550,4
shipments,actual_delivery_date,89681,2321
shipments,delivery_days,89681,2321


In [0]:

from pyspark.sql import functions as F

# Numeric column definitions
numeric_columns = {
    "orders": {
        "discount_percentage": "DOUBLE",
        "subtotal": "DOUBLE",
        "shipping_fee": "DOUBLE",
        "tax_amount": "DOUBLE",
        "final_amount": "DOUBLE"
    },

    "order_items": {
        "quantity": "INT",
        "unit_price": "DOUBLE",
        "discount_percentage": "DOUBLE",
        "item_revenue": "DOUBLE",
        "item_cost": "DOUBLE",
        "profit": "DOUBLE"
    },

    "customers": {
        "age": "INT",
        "pincode_prefix": "INT",
        "total_orders": "INT",
        "total_spend": "DOUBLE",
        "average_order_value": "DOUBLE"
    },

    "products": {
        "price": "DOUBLE",
        "cost_price": "DOUBLE",
        "rating_average": "DOUBLE",
        "rating_count": "INT",
        "stock_quantity": "INT",
        "return_rate_baseline": "DOUBLE"
    },

    "payments": {
        "amount_paid": "DOUBLE",
        "transaction_fee": "DOUBLE",
        "refund_amount": "DOUBLE"
    },

    "shipments": {
        "delivery_days": "INT",
        "delayed_flag": "INT"
    },

    "returns": {
        "refund_amount": "DOUBLE"
    },

    "customer_reviews": {
        "rating": "INT"
    },

    "marketing_campaigns": {
        "discount_percentage": "INT",
        "conversions": "INT",
        "revenue_generated": "DOUBLE",
        "impressions": "BIGINT",
        "clicks": "BIGINT",
        "campaign_cost": "DOUBLE",
        "roi": "DOUBLE"
    }
}

# Date column definitions
date_columns = {
    "orders": ["order_date"],
    "customers": ["customer_signup_date", "last_order_date"],
    "products": ["product_launch_date"],
    "payments": ["payment_date"],
    "shipments": [
        "dispatch_date",
        "expected_delivery_date",
        "actual_delivery_date"
    ],
    "returns": ["return_date"],
    "customer_reviews": ["review_date"],
    "marketing_campaigns": [
        "campaign_start_date",
        "campaign_end_date"
    ]
}

# Boolean columns
boolean_columns = {
    "customer_reviews": ["verified_purchase"]
}

print("Silver data type mappings configured.")


Silver data type mappings configured.


In [0]:

silver_tables = {}

for table_name in TABLES:

    print(f"Transforming: {table_name}")

    df = bronze_tables[table_name]

    # Convert numeric columns safely
    for column_name, data_type in numeric_columns.get(
        table_name, {}
    ).items():

        df = df.withColumn(
            column_name,
            F.expr(
                f"try_cast(`{column_name}` AS {data_type})"
            )
        )

    # Convert date columns safely
    for column_name in date_columns.get(table_name, []):

        df = df.withColumn(
            column_name,
            F.expr(
                f"try_cast(`{column_name}` AS DATE)"
            )
        )

    # Convert boolean columns
    for column_name in boolean_columns.get(table_name, []):

        df = df.withColumn(
            column_name,
            F.when(
                F.lower(F.trim(F.col(column_name))).isin(
                    "true", "1", "yes"
                ),
                F.lit(True)
            )
            .when(
                F.lower(F.trim(F.col(column_name))).isin(
                    "false", "0", "no"
                ),
                F.lit(False)
            )
            .otherwise(F.lit(None).cast("BOOLEAN"))
        )

    # Add transformation metadata
    df = df.withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )

    # Write Silver Delta table
    target_table = f"{SILVER_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    silver_tables[table_name] = spark.table(target_table)

    print(
        f"Completed: {table_name} | "
        f"Rows: {silver_tables[table_name].count()}"
    )

print("\nSilver data type standardization completed.")


Transforming: orders
Completed: orders | Rows: 100000
Transforming: order_items
Completed: order_items | Rows: 254331
Transforming: customers
Completed: customers | Rows: 25000
Transforming: products
Completed: products | Rows: 550
Transforming: payments
Completed: payments | Rows: 100000
Transforming: shipments
Completed: shipments | Rows: 89681
Transforming: returns
Completed: returns | Rows: 12075
Transforming: customer_reviews
Completed: customer_reviews | Rows: 77530
Transforming: marketing_campaigns
Completed: marketing_campaigns | Rows: 151

Silver data type standardization completed.


In [0]:

# Inspect transformed Silver schemas

for table_name, df in silver_tables.items():

    print(f"\n{'=' * 50}")
    print(f"SILVER TABLE: {table_name.upper()}")
    print(f"{'=' * 50}")

    df.printSchema()



SILVER TABLE: ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_time: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_method: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- shipping_fee: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)


SILVER TABLE: ORDER_ITEMS
root
 |-- order_item_id: string (nullable = tru

In [0]:

# Compare Bronze and Silver record counts

comparison = []

for table_name in TABLES:

    bronze_count = bronze_tables[table_name].count()
    silver_count = silver_tables[table_name].count()

    comparison.append((
        table_name,
        bronze_count,
        silver_count,
        silver_count == bronze_count
    ))

comparison_df = spark.createDataFrame(
    comparison,
    [
        "table_name",
        "bronze_count",
        "silver_count",
        "count_match"
    ]
)

display(comparison_df)


table_name,bronze_count,silver_count,count_match
orders,100000,100000,true
order_items,254331,254331,true
customers,25000,25000,true
products,550,550,true
payments,100000,100000,true
shipments,89681,89681,true
returns,12075,12075,true
customer_reviews,77530,77530,true
marketing_campaigns,151,151,true


In [0]:

from pyspark.sql import functions as F
from functools import reduce

# Primary keys for all nine tables
PRIMARY_KEYS = {
    "orders": ["order_id"],
    "order_items": ["order_item_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "payments": ["payment_id"],
    "shipments": ["shipment_id"],
    "returns": ["return_id"],
    "customer_reviews": ["review_id"],
    "marketing_campaigns": ["campaign_id"]
}

# Store validation results
duplicate_results = []
quarantine_tables = {}

for table_name, primary_key in PRIMARY_KEYS.items():

    print(f"Validating: {table_name}")

    df = silver_tables[table_name]

    # Identify records with missing primary keys
    null_condition = reduce(
        lambda a, b: a | b,
        [F.col(key).isNull() for key in primary_key]
    )

    null_pk_df = df.filter(null_condition)

    # Identify duplicate primary-key records
    duplicate_keys = (
        df.filter(~null_condition)
        .groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .select(*primary_key)
    )

    duplicate_df = (
        df.join(duplicate_keys, primary_key, "inner")
    )

    # Collect validation counts
    null_pk_count = null_pk_df.count()
    duplicate_record_count = duplicate_df.count()

    duplicate_results.append((
        table_name,
        null_pk_count,
        duplicate_record_count
    ))

    # Combine invalid records
    quarantine_df = (
        null_pk_df
        .unionByName(duplicate_df)
        .dropDuplicates()
        .withColumn("_quarantine_reason", F.lit("INVALID_PRIMARY_KEY"))
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    quarantine_tables[table_name] = quarantine_df

    print(
        f"{table_name}: "
        f"Null PKs={null_pk_count}, "
        f"Duplicate records={duplicate_record_count}"
    )

print("Primary-key validation completed.")


Validating: orders
orders: Null PKs=0, Duplicate records=0
Validating: order_items
order_items: Null PKs=0, Duplicate records=0
Validating: customers
customers: Null PKs=0, Duplicate records=0
Validating: products
products: Null PKs=0, Duplicate records=0
Validating: payments
payments: Null PKs=0, Duplicate records=0
Validating: shipments
shipments: Null PKs=0, Duplicate records=0
Validating: returns
returns: Null PKs=0, Duplicate records=0
Validating: customer_reviews
customer_reviews: Null PKs=0, Duplicate records=0
Validating: marketing_campaigns
marketing_campaigns: Null PKs=0, Duplicate records=0
Primary-key validation completed.


In [0]:

duplicate_report = spark.createDataFrame(
    duplicate_results,
    [
        "table_name",
        "null_primary_key_records",
        "duplicate_primary_key_records"
    ]
)

display(
    duplicate_report.orderBy("table_name")
)


table_name,null_primary_key_records,duplicate_primary_key_records
customer_reviews,0,0
customers,0,0
marketing_campaigns,0,0
order_items,0,0
orders,0,0
payments,0,0
products,0,0
returns,0,0
shipments,0,0


In [0]:

from pyspark.sql import functions as F

# Define business validation rules
BUSINESS_RULES = {
    "orders": [
        ("final_amount", "final_amount < 0"),
        ("discount_percentage", "discount_percentage < 0 OR discount_percentage > 100")
    ],
    "order_items": [
        ("quantity", "quantity <= 0"),
        ("unit_price", "unit_price < 0"),
        ("discount_percentage", "discount_percentage < 0 OR discount_percentage > 100")
    ],
    "products": [
        ("price", "price < 0"),
        ("cost_price", "cost_price < 0"),
        ("rating_average", "rating_average < 0 OR rating_average > 5"),
        ("stock_quantity", "stock_quantity < 0")
    ],
    "payments": [
        ("amount_paid", "amount_paid < 0"),
        ("transaction_fee", "transaction_fee < 0"),
        ("refund_amount", "refund_amount < 0")
    ],
    "shipments": [
        ("delivery_days", "delivery_days < 0"),
        ("delayed_flag", "delayed_flag NOT IN (0, 1)")
    ],
    "customer_reviews": [
        ("rating", "rating < 1 OR rating > 5")
    ],
    "marketing_campaigns": [
        ("clicks", "clicks > impressions"),
        ("conversions", "conversions < 0"),
        ("campaign_cost", "campaign_cost < 0")
    ]
}

business_results = []

for table_name, rules in BUSINESS_RULES.items():

    df = silver_tables[table_name]

    for column_name, condition in rules:

        invalid_count = df.filter(
            F.expr(condition)
        ).count()

        business_results.append((
            table_name,
            column_name,
            condition,
            invalid_count
        ))

        print(
            f"{table_name}.{column_name}: "
            f"{invalid_count} invalid records"
        )

business_report = spark.createDataFrame(
    business_results,
    [
        "table_name",
        "column_name",
        "validation_rule",
        "invalid_records"
    ]
)

display(business_report)


orders.final_amount: 0 invalid records
orders.discount_percentage: 0 invalid records
order_items.quantity: 0 invalid records
order_items.unit_price: 0 invalid records
order_items.discount_percentage: 0 invalid records
products.price: 0 invalid records
products.cost_price: 0 invalid records
products.rating_average: 0 invalid records
products.stock_quantity: 0 invalid records
payments.amount_paid: 0 invalid records
payments.transaction_fee: 0 invalid records
payments.refund_amount: 0 invalid records
shipments.delivery_days: 0 invalid records
shipments.delayed_flag: 0 invalid records
customer_reviews.rating: 0 invalid records
marketing_campaigns.clicks: 0 invalid records
marketing_campaigns.conversions: 0 invalid records
marketing_campaigns.campaign_cost: 0 invalid records


table_name,column_name,validation_rule,invalid_records
orders,final_amount,final_amount < 0,0
orders,discount_percentage,discount_percentage < 0 OR discount_percentage > 100,0
order_items,quantity,quantity <= 0,0
order_items,unit_price,unit_price < 0,0
order_items,discount_percentage,discount_percentage < 0 OR discount_percentage > 100,0
products,price,price < 0,0
products,cost_price,cost_price < 0,0
products,rating_average,rating_average < 0 OR rating_average > 5,0
products,stock_quantity,stock_quantity < 0,0
payments,amount_paid,amount_paid < 0,0


In [0]:

from pyspark.sql import functions as F

# Child table, child column, parent table, parent column
RELATIONSHIPS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "campaign_id", "marketing_campaigns", "campaign_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),
    ("returns", "order_id", "orders", "order_id"),
    ("returns", "customer_id", "customers", "customer_id"),
    ("returns", "product_id", "products", "product_id"),
    ("customer_reviews", "order_id", "orders", "order_id"),
    ("customer_reviews", "customer_id", "customers", "customer_id"),
    ("customer_reviews", "product_id", "products", "product_id")
]

fk_results = []

for child, child_col, parent, parent_col in RELATIONSHIPS:

    child_df = silver_tables[child]
    parent_df = silver_tables[parent]

    parent_keys = (
        parent_df
        .select(parent_col)
        .filter(F.col(parent_col).isNotNull())
        .distinct()
    )

    child_values = child_df.select(child_col)

    orphan_count = (
        child_values
        .filter(F.col(child_col).isNotNull())
        .join(
            parent_keys,
            child_values[child_col] == parent_keys[parent_col],
            "left_anti"
        )
        .count()
    )

    null_count = child_df.filter(
        F.col(child_col).isNull()
    ).count()

    fk_results.append((
        child,
        child_col,
        parent,
        orphan_count,
        null_count,
        "PASS" if orphan_count == 0 else "REVIEW"
    ))

fk_report = spark.createDataFrame(
    fk_results,
    [
        "child_table",
        "foreign_key",
        "parent_table",
        "orphan_records",
        "null_records",
        "status"
    ]
)

display(fk_report)


child_table,foreign_key,parent_table,orphan_records,null_records,status
orders,customer_id,customers,0,0,PASS
orders,campaign_id,marketing_campaigns,0,82821,PASS
order_items,order_id,orders,0,0,PASS
order_items,product_id,products,0,0,PASS
payments,order_id,orders,0,0,PASS
shipments,order_id,orders,0,0,PASS
returns,order_id,orders,0,0,PASS
returns,customer_id,customers,0,0,PASS
returns,product_id,products,0,0,PASS
customer_reviews,order_id,orders,0,0,PASS


In [0]:

# Final Silver table count verification

final_counts = []

for table_name in TABLES:

    bronze_count = bronze_tables[table_name].count()
    silver_count = silver_tables[table_name].count()

    final_counts.append((
        table_name,
        bronze_count,
        silver_count,
        "PASS" if bronze_count == silver_count else "REVIEW"
    ))

final_report = spark.createDataFrame(
    final_counts,
    [
        "table_name",
        "bronze_records",
        "silver_records",
        "status"
    ]
)

display(final_report)

# Save quality reports in the Silver schema
(
    fk_report.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{SILVER_SCHEMA}.foreign_key_validation"
    )
)

(
    final_report.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{SILVER_SCHEMA}.record_count_validation"
    )
)

print("Silver quality reports saved successfully.")


table_name,bronze_records,silver_records,status
orders,100000,100000,PASS
order_items,254331,254331,PASS
customers,25000,25000,PASS
products,550,550,PASS
payments,100000,100000,PASS
shipments,89681,89681,PASS
returns,12075,12075,PASS
customer_reviews,77530,77530,PASS
marketing_campaigns,151,151,PASS


Silver quality reports saved successfully.


In [0]:
display(spark.sql(
    "SHOW TABLES IN online_retail.indian_ecommerce_silver"
))

database,tableName,isTemporary
indian_ecommerce_silver,customer_reviews,false
indian_ecommerce_silver,customers,false
indian_ecommerce_silver,foreign_key_validation,false
indian_ecommerce_silver,marketing_campaigns,false
indian_ecommerce_silver,order_items,false
indian_ecommerce_silver,orders,false
indian_ecommerce_silver,payments,false
indian_ecommerce_silver,products,false
indian_ecommerce_silver,record_count_validation,false
indian_ecommerce_silver,returns,false
